In [1]:
# suppressMessages(library(ArchR))
suppressMessages(library(Seurat))
suppressMessages(library(dplyr))
suppressMessages(library(igraph))
suppressMessages(library(ggraph))
suppressMessages(library(harmony))
library(igraph) 
library(qs) 
library(ggsci)

suppressMessages(library(reticulate))
RETICULATE_PYTHON <- "/home/chaiqw/project/mb/.venv/bin/python"
use_virtualenv("/home/chaiqw/project/mb/.venv")

plan("multicore", workers = 32)
options(future.globals.maxSize = 100000 * 1024^5)



qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103



In [2]:
seurat1 = readRDS('/mnt/90-connectome/Personal/CQW/human/snRNA/cross_species_seurat1_Astro_sub_downsampled_region0.3-unifygene-20260225.rds')
seurat2 = readRDS('/mnt/90-connectome/Personal/CQW/human/snRNA/cross_species_seurat2_Astro_sub_downsampled_region0.7-unifygene-20260225.rds')
seurat3 = readRDS('/mnt/90-connectome/Personal/CQW/human/snRNA/cross_species_seurat3_Astro_sub_downsampled_region0.35-unifygene-20260225.rds')


In [37]:
seurat1
seurat2
seurat3

An object of class Seurat 
33304 features across 106198 samples within 1 assay 
Active assay: RNA (33304 features, 0 variable features)
 1 layer present: counts

An object of class Seurat 
58225 features across 108518 samples within 1 assay 
Active assay: RNA (58225 features, 0 variable features)
 1 layer present: counts

An object of class Seurat 
32245 features across 108038 samples within 1 assay 
Active assay: RNA (32245 features, 0 variable features)
 1 layer present: counts

In [50]:
seurat1_sub = subset(seurat1, downsample = 1000)
Idents(seurat2) <- ''
seurat2_sub = subset(seurat2, downsample = 1000)
seurat3_sub = subset(seurat3, downsample = 1000)

In [3]:
seurat1_sub = seurat1
seurat2_sub = seurat2
seurat3_sub = seurat3

In [4]:
seurat1_sub$batch = 'seurat1'
seurat2_sub$batch = 'seurat2'
seurat3_sub$batch = 'seurat3'

In [5]:
# 提取三个数据集的共有基因，并只保留共有基因的表达矩阵
common_genes <- Reduce(intersect, list(rownames(seurat1_sub), rownames(seurat2_sub), rownames(seurat3_sub)))
seurat1_sub <- subset(seurat1_sub, features = common_genes)
seurat2_sub <- subset(seurat2_sub, features = common_genes)
seurat3_sub <- subset(seurat3_sub, features = common_genes)

In [10]:
seurat1_sub
seurat2_sub
seurat3_sub

An object of class Seurat 
15980 features across 106198 samples within 1 assay 
Active assay: RNA (15980 features, 0 variable features)
 1 layer present: counts

An object of class Seurat 
15980 features across 108518 samples within 1 assay 
Active assay: RNA (15980 features, 0 variable features)
 1 layer present: counts

An object of class Seurat 
15980 features across 108038 samples within 1 assay 
Active assay: RNA (15980 features, 0 variable features)
 1 layer present: counts

In [6]:

seurat1_sub_1 <- seurat1_sub %>% 
    NormalizeData(normalization.method = "LogNormalize") %>% 
    FindVariableFeatures( nfeatures = 2000) %>% 
    ScaleData() %>% 
    RunPCA(npcs=50,verbose = FALSE) %>% 
    FindNeighbors(dims = 1:20) %>%
    FindClusters(verbose = FALSE, resolution = 0.5, random.seed = 1234) %>%
    RunUMAP(dims = 1:20, seed.use = 1234)

Normalizing layer: counts

Finding variable features for layer counts

Centering and scaling data matrix

Computing nearest neighbor graph

Computing SNN

Warning message:
“UNRELIABLE VALUE: One of the ‘future.apply’ iterations (‘future_lapply-1’) unexpectedly generated random numbers without declaring so. There is a risk that those random numbers are not statistically sound and the overall results might be invalid. To fix this, specify 'future.seed=TRUE'. This ensures that proper, parallel-safe random numbers are produced via a parallel RNG method. To disable this check, use 'future.seed = NULL', or set option 'future.rng.onMisuse' to "ignore".”
Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”
13:34:35 UMAP embedding parameters a = 0.9922 b = 1.112

13

In [7]:
seurat2_sub_1 <- seurat2_sub %>% 
    NormalizeData(normalization.method = "LogNormalize") %>% 
    FindVariableFeatures( nfeatures = 2000) %>% 
    ScaleData() %>% 
    RunPCA(npcs=50,verbose = FALSE) %>% 
    FindNeighbors(dims = 1:20) %>%
    FindClusters(verbose = FALSE, resolution = 0.5, random.seed = 1234) %>%
    RunUMAP(dims = 1:20, seed.use = 1234)

Normalizing layer: counts

Finding variable features for layer counts

Centering and scaling data matrix

Computing nearest neighbor graph

Computing SNN

Warning message:
“UNRELIABLE VALUE: One of the ‘future.apply’ iterations (‘future_lapply-1’) unexpectedly generated random numbers without declaring so. There is a risk that those random numbers are not statistically sound and the overall results might be invalid. To fix this, specify 'future.seed=TRUE'. This ensures that proper, parallel-safe random numbers are produced via a parallel RNG method. To disable this check, use 'future.seed = NULL', or set option 'future.rng.onMisuse' to "ignore".”
13:38:39 UMAP embedding parameters a = 0.9922 b = 1.112

13:38:39 Read 108518 rows and found 20 numeric columns

13:38:39 Using Annoy for neighbor search, n_neighbors = 30

13:38:39 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----

In [8]:
seurat3_sub_1 <- seurat3_sub %>% 
    NormalizeData(normalization.method = "LogNormalize") %>% 
    FindVariableFeatures( nfeatures = 2000) %>% 
    ScaleData() %>% 
    RunPCA(npcs=50,verbose = FALSE) %>% 
    FindNeighbors(dims = 1:20) %>%
    FindClusters(verbose = FALSE, resolution = 0.5, random.seed = 1234) %>%
    RunUMAP(dims = 1:20, seed.use = 1234)

Normalizing layer: counts

Finding variable features for layer counts

Centering and scaling data matrix

Computing nearest neighbor graph

Computing SNN

Warning message:
“UNRELIABLE VALUE: One of the ‘future.apply’ iterations (‘future_lapply-1’) unexpectedly generated random numbers without declaring so. There is a risk that those random numbers are not statistically sound and the overall results might be invalid. To fix this, specify 'future.seed=TRUE'. This ensures that proper, parallel-safe random numbers are produced via a parallel RNG method. To disable this check, use 'future.seed = NULL', or set option 'future.rng.onMisuse' to "ignore".”
13:43:25 UMAP embedding parameters a = 0.9922 b = 1.112

13:43:25 Read 108038 rows and found 20 numeric columns

13:43:25 Using Annoy for neighbor search, n_neighbors = 30

13:43:25 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----

In [40]:
length(common_genes1)

[1] 4472

In [9]:
# # 将对象放入list
obj_list <- list(
  seurat1 = seurat1_sub_1,
  seurat2 = seurat2_sub_1,
  seurat3 = seurat3_sub_1
)
# 计算三个对象的高变基因并集
common_genes1 <- unique(c(
  VariableFeatures(seurat1_sub_1),
  VariableFeatures(seurat2_sub_1),
  VariableFeatures(seurat3_sub_1)
))

# # 找到锚点
anchors <- FindIntegrationAnchors(object.list = obj_list, anchor.features = common_genes1, reduction = "cca")

# 整合数据
seurat_cca_integrated <- IntegrateData(
  normalization.method = "LogNormalize",
  anchorset = anchors,
  features.to.integrate = common_genes1
)


Scaling features for provided objects

Warning message:
“Different features in new layer data than already exists for scale.data”
Warning message:
“Different features in new layer data than already exists for scale.data”
Warning message:
“Different features in new layer data than already exists for scale.data”
Finding all pairwise anchors

Warning message in mccollect(jobs = jobs, wait = TRUE):
“1 parallel job did not deliver a result”
Warning message:
“Caught FutureInterruptError. Canceling all iterations ...”


ERROR: Error: A future (‘future_lapply-3’) of class MulticoreFuture was interrupted, while running on localhost (pid 2121644)


In [ ]:
seurat_cca_integrated

An object of class Seurat 
31960 features across 3000 samples within 2 assays 
Active assay: integrated (15980 features, 15980 variable features)
 1 layer present: data
 1 other assay present: RNA

In [ ]:
seurat_cca_integrated_1 <- seurat_cca_integrated %>% 
    ScaleData(assay = "integrated",features = common_genes1) %>%    
    RunPCA(npcs = 50, assay = "integrated",verbose = F) %>% 
    FindNeighbors(dims = 1:20,reduction = "pca",k.param = 40) %>%
    FindClusters(resolution = 3,random.seed = 123)%>%
    RunUMAP(dims = 1:20 ,assay = "integrated",min.dist = 0.3)

ERROR: Error: 找不到对象'seurat_cca_integrated'


In [ ]:
# Idents(seurat_cca_integrated_1) <- 'celltype_new'
options(repr.plot.width=12, repr.plot.height=4)

# 使用ggsci的pal_nejm配色方案（再换一个配色）
DimPlot(
  seurat_cca_integrated_1, 
  split.by = "batch",  
  label = TRUE,
  pt.size = 0.001
) & labs(title = "V5 Integ")

ERROR: Error: 找不到对象'seurat_cca_integrated_1'
